In [2]:
import numpy as np

## 4.손실함수

In [3]:
def sum_squares_error(y, t):        ## MSE랑 같은거 아닌가
    return 0.5 * np.sum((y-t)**2)

In [4]:
t = [0,0,1,0,0,0,0,0,0,0]
y = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0]
print(sum_squares_error(np.array(y), np.array(t)))

y = [0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0]
print(sum_squares_error(np.array(y), np.array(t)))

0.09750000000000003
0.5975


In [5]:
def cross_entropy_error(y, t):
    return -np.sum(t * np.log(y + 1e-7))

In [6]:
t = [0,0,1,0,0,0,0,0,0,0]
y = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0]
print(cross_entropy_error(np.array(y), np.array(t)))

y = [0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0]
print(cross_entropy_error(np.array(y), np.array(t)))

0.510825457099338
2.302584092994546


## 미니배치 학습
+ 원 핫 라벨을 쓰는 이유?
  + 정답은 0, 정답이 아닌 걸 1로 해야지 손실을 계산하기 쉽겠네

In [7]:
import sys, os

if './official_github' not in sys.path:
    sys.path.append('./official_github')

In [8]:
from dataset.mnist import load_mnist

(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

In [9]:
batch_mask = np.random.choice(train_size:=x_train.shape[0], batch_size:=10)
print(batch_mask, type(batch_mask), sep='\n')

x_batch = x_train[batch_mask]
t_batch = t_train[batch_mask]

[49681 18664 43755 55684 24341 57866  2060 43603 11647 32404]
<class 'numpy.ndarray'>


In [10]:
def cross_entropy_error(y, t):
    if y.ndim == 1:  ## no mini-batch!
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
    
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size


In [11]:
# ## one-hot encoding이 **아닌** 경우
# def cross_entropy_error(y, t):
#     if y.ndim == 1:  ## no mini-batch!
#         t = t.reshape(1, t.size)
#         y = y.reshape(1, y.size)
    
#     batch_size = y.shape[0]
#     return -np.sum(t * np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size


## 4.3.수치 미분

In [12]:
def numerical_diff(f,x):
    h = 1e-4
    return (f(x+h) - f(x-h)) / 2*h

def function_1(x):
    return 0.01*x**2 + 0.1*x

print(numerical_diff(function_1, 10))

2.999999999986347e-09


In [13]:
def function_2(x: np.ndarray):
    return x[0]**2 + x[1]**2

## 편미분
def numerical_grad(f, x):
    h = 1e-4
    grad = np.zeros_like(x)
    
    for idx in range(x.size):
        tmp = x[idx]    ## 값 복원을 위해 저장
        
        ## f(x+h)
        x[idx] = tmp + h
        fxh1 = f(x)
        
        ## f(x-h)
        x[idx] = tmp - h
        fxh2 = f(x)
        grad[idx] = (fxh1 - fxh2) / (2*h)
        x[idx] = tmp    ## 값 복원
    
    return grad
        
numerical_grad(function_2, np.array([3.0, 4.0]))

array([6., 8.])

In [ ]:
g1 = function_2(np.array([3.0 + 1e-4, 4.0]))

In [ ]:
g2 = function_2(np.array([3.0 - 1e-4, 4.0]))

In [ ]:
(g1 - g2) / (2 * 1e-4)

## 경사하강법

In [ ]:
def gradient_descent(f, init_x, lr=0.01, step_num=100):
    x = init_x
    
    for i in range(step_num):
        grad = numerical_grad(f,x)
        x -= lr * grad
        print(f'{i}th grad: {x}')
    
    return x

In [ ]:
gradient_descent(function_2, np.array([-3.0, 4.0]))

In [ ]:
## lr이 너무 적으면 학습은 되는데...아주아주 느리게 되는구만
gradient_descent(function_2, np.array([-3.0, 4.0]), lr=1e-10, step_num = 100000)


### 4.4.2.신경망에서의 기울기

In [ ]:
import sys, os

if './official_github/' not in sys.path:
    sys.path.append('./official_github/')    

import numpy as np
from common.functions import softmax, cross_entropy_error
from common.gradient import numerical_gradient

In [ ]:
class SimpleNet:
    def __init__(self):
        self.W = np.random.randn(2,3)
    
    def predict(self, x):
        return x @ self.W

    def loss(self, x, t):
        z = self.predict(x)
        y = softmax(z)
        loss = cross_entropy_error(y, t)
        
        return loss

In [ ]:
net = SimpleNet()
print(net.W)

x = np.array([0.6, 0.9])
print(x)
print(p:= net.predict(x))
np.argmax(p)


In [ ]:
def f(W):
    return net.loss(x, t)

In [ ]:
dW = numerical_grad(f,net.W)
print(dW)

## 4.5.학습 알고리즘 구현하기
> hidden size는 얼마로 해야 좋은가???

In [ ]:
import sys, os

if './official_github/' not in sys.path:
    sys.path.append('./official_github/')    

import numpy as np
from common.functions import sigmoid, softmax, cross_entropy_error
from common.gradient import numerical_gradient


In [ ]:
class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size_size, output_size)
        self.params['b1'] = np.zeros(output_size)
        
    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        
        a1 = x @ W1 + b1
        z1 = sigmoid(a1)
        a2 = z1 @ W2 + b2
        y = softmax(a2)
    
    def loss(self, x, t):
        y = self.predict(x)
        
        return cross_entropy_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        t = np.argmax(t, axis=1)
        
        accuracy = np.sum(y == t) / float(x.shape[0])
        
        return accuracy
    
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x. t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
    
        return grads
        
        

### 4.2.4.미니배치 학습 구현하기

In [ ]:
import numpy as np
from dataset.mnist import load_mnist
from two_layer_net import TwoLayerNet

In [ ]:
t = np.random.randn(5,5)
print(t)
n = t[np.array([3,4]),:]
w = t[[3,4]]
print(n, n.shape)
print(w, w.shape)

In [ ]:
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)
print(x_train.shape)

train_loss_list = []
train_acc_list = []
test_acc_list = []


iters_num = 1000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

iter_per_epochs = max(train_size / batch_size, 1)

net = TwoLayerNet(input_size = 784, hidden_size=50, output_size=10)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)    ## 784개의 집단에서 100개많큼 뽑는다.
    x_batch = x_train[batch_mask, :]
    t_batch = t_train[batch_mask, :]
    
    grad = net.numerical_gradient(x_batch, t_batch)
    
    for key in net.params:
        net.params[key] -= learning_rate * grad[key]
    
    loss = net.loss(x_batch, t_batch)
    train_loss_list.append(loss)

In [71]:
train_loss_list

[2.294289571910697,
 2.2966110092016208,
 2.2855266305117574,
 2.276737440438543,
 2.2973183055753745,
 2.295280244223511]